In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import joblib

In [3]:
df = pd.read_csv('../data/train.csv')
 
df = df.replace('NaN ', np.nan).replace(' NaN', np.nan)

In [4]:
df['Delivery_person_Age'] = pd.to_numeric(df['Delivery_person_Age'], errors='coerce')
df['Delivery_person_Ratings'] = pd.to_numeric(df['Delivery_person_Ratings'], errors='coerce')
df['multiple_deliveries'] = pd.to_numeric(df['multiple_deliveries'], errors='coerce')
df['Time_taken(min)'] = df['Time_taken(min)'].str.replace('(min) ', '', regex=False).astype(float)
df['Weatherconditions'] = df['Weatherconditions'].str.replace('conditions ', '', regex=False).str.strip()
df['Road_traffic_density'] = df['Road_traffic_density'].str.strip()
df['Festival'] = df['Festival'].str.strip()
df['City'] = df['City'].str.strip()
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d-%m-%Y', errors='coerce')
 
df['Delivery_person_Age'] = df['Delivery_person_Age'].fillna(df['Delivery_person_Age'].median())
df['Delivery_person_Ratings'] = df['Delivery_person_Ratings'].fillna(df['Delivery_person_Ratings'].median())
df['multiple_deliveries'] = df['multiple_deliveries'].fillna(df['multiple_deliveries'].mode()[0])
df['Road_traffic_density'] = df['Road_traffic_density'].fillna(df['Road_traffic_density'].mode()[0])
df['Festival'] = df['Festival'].fillna(df['Festival'].mode()[0])
df['City'] = df['City'].fillna(df['City'].mode()[0])
df['Time_Orderd'] = df['Time_Orderd'].fillna(df['Time_Order_picked'])
 

In [5]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))
 
df['distance_km'] = df.apply(lambda r: haversine(
    r['Restaurant_latitude'], r['Restaurant_longitude'],
    r['Delivery_location_latitude'], r['Delivery_location_longitude']), axis=1)
 
df['order_hour'] = pd.to_datetime(df['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.hour
df['day_of_week'] = df['Order_Date'].dt.dayofweek
df['is_peak_hour'] = df['order_hour'].apply(lambda h: 1 if h in [12, 13, 14, 19, 20, 21] else 0)
 
df = df[(df['Time_taken(min)'] >= 10) & (df['Time_taken(min)'] <= 60)]

In [8]:
# eporter en excel pour analyse
df.to_excel('../data/cleaned_train.xlsx', index=False)


In [6]:
colonnes_a_supprimer = [
    'ID', 'Delivery_person_ID', 'Order_Date', 'Time_Orderd',
    'Time_Order_picked', 'Restaurant_latitude', 'Restaurant_longitude',
    'Delivery_location_latitude', 'Delivery_location_longitude'
]
df = df.drop(columns=colonnes_a_supprimer)
 
cat_cols = ['Weatherconditions', 'Road_traffic_density', 'Type_of_order',
            'Type_of_vehicle', 'Festival', 'City']
 
le = LabelEncoder()
for col in cat_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

df = df.dropna()
 
X = df.drop(columns=['Time_taken(min)'])
y = df['Time_taken(min)']
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
 
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
 
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
 
print(f"Régression Linéaire → MAE: {mae_lr:.2f} min | RMSE: {rmse_lr:.2f} min")
print(f"Random Forest       → MAE: {mae_rf:.2f} min | RMSE: {rmse_rf:.2f} min")
print(f"Amélioration        → {((mae_lr - mae_rf)/mae_lr*100):.1f}%")
 
joblib.dump(rf, 'model_random_forest.pkl')
 

Régression Linéaire → MAE: 5.38 min | RMSE: 6.73 min
Random Forest       → MAE: 3.22 min | RMSE: 4.05 min
Amélioration        → 40.2%


['model_random_forest.pkl']